<a href="https://colab.research.google.com/github/evakonstantinova/EfficientNet-B0/blob/main/HQNN_IBM_Calibration_Noise_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# MANUALLY UPLOAD THE TRAINED BEST_HQNN CHECKPOINT

from google.colab import files
import os

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError(
        "Please upload exactly one trained HQNN .pth checkpoint file."
    )

uploaded_filename = next(iter(uploaded))

if not uploaded_filename.lower().endswith(".pth"):
    raise ValueError(
        f"Expected a .pth checkpoint, but received: {uploaded_filename}"
    )

HQNN_CHECKPOINT = f"/content/{uploaded_filename}"

if not os.path.exists(HQNN_CHECKPOINT):
    raise FileNotFoundError(
        f"Uploaded checkpoint was not found at {HQNN_CHECKPOINT}"
    )

print("HQNN checkpoint uploaded successfully.")
print("Filename:", uploaded_filename)
print("Path:", HQNN_CHECKPOINT)
print(
    "Size:",
    round(os.path.getsize(HQNN_CHECKPOINT) / (1024 ** 2), 2),
    "MB"
)


Saving BEST_HQNN (1).pth to BEST_HQNN (1).pth
HQNN checkpoint uploaded successfully.
Filename: BEST_HQNN (1).pth
Path: /content/BEST_HQNN (1).pth
Size: 15.59 MB


In [2]:
# INSTALL THE REQUIRED, COMPATIBLE QUANTUM SOFTWARE VERSIONS

# PennyLane-Qiskit is intentionally NOT installed because this notebook
# uses PennyLane default.qubit and Qiskit Aer directly.
!pip uninstall -y qiskit qiskit-aer qiskit-ibm-runtime > /dev/null 2>&1

!pip install -q \
    "pennylane==0.45.1" \
    "qiskit==2.3.0" \
    "qiskit-aer==0.17.2" \
    "qiskit-ibm-runtime==0.45.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.

In [3]:
# IMPORT LIBRARIES AND SET REPRODUCIBILITY CONTROLS

import json
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import pennylane as qml

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("PyTorch version:", torch.__version__)
print("PennyLane version:", qml.__version__)
print("Random seed fixed:", SEED)


PyTorch version: 2.11.0+cu128
PennyLane version: 0.45.1
Random seed fixed: 42


In [4]:
# DOWNLOAD THE MRI DATASET AND RECREATE THE SAME 70/15/15 SPLIT

from pathlib import Path
from collections import Counter

import kagglehub
from sklearn.model_selection import train_test_split

path = kagglehub.dataset_download(
    "masoudnickparvar/brain-tumor-mri-dataset"
)

classes = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]

all_files = []
all_labels = []

for class_name in classes:
    for folder in ["Training", "Testing"]:

        class_path = (
            Path(path)
            / folder
            / class_name
        )

        # IMPORTANT: keep the same file-enumeration procedure used in
        # the noise-free HQNN training notebook so random_state=42
        # recreates the same split logic as closely as possible.
        for file_path in class_path.iterdir():

            if file_path.is_file():
                all_files.append(str(file_path))
                all_labels.append(class_name)

train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files,
    all_labels,
    test_size=0.30,
    random_state=42,
    stratify=all_labels
)

val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

print("Dataset path:", path)
print("Total images:", len(all_files))
print("Overall distribution:", Counter(all_labels))
print("Test images:", len(test_files))
print("Test distribution:", Counter(test_labels))


Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Dataset path: /kaggle/input/brain-tumor-mri-dataset
Total images: 7200
Overall distribution: Counter({'glioma': 1800, 'meningioma': 1800, 'notumor': 1800, 'pituitary': 1800})
Test images: 1080
Test distribution: Counter({'meningioma': 270, 'notumor': 270, 'glioma': 270, 'pituitary': 270})


In [5]:
# CREATE THE FIXED TEST DATASET AND DATALOADER

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class_to_idx = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class BrainTumorDataset(Dataset):

    def __init__(
        self,
        files,
        labels,
        transform=None
    ):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        image = Image.open(
            self.files[idx]
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[
            self.labels[idx]
        ]

        return image, label


test_dataset = BrainTumorDataset(
    test_files,
    test_labels,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

assert len(test_dataset) == 1080, (
    f"Expected 1,080 test images, found {len(test_dataset)}."
)

print("Test dataset ready:", len(test_dataset))


Test dataset ready: 1080


In [6]:
# DEFINING THE QUANTUM CIRCUIT

# DEFINE THE QUANTUM CIRCUIT

N_QUBITS = 4
N_Q_LAYERS = 2

quantum_device = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

@qml.qnode(
    quantum_device,
    interface="torch",
    diff_method="backprop"
)
def quantum_circuit(inputs, weights):

    # ENCODE 4 CLASSICAL FEATURES INTO 4 QUBITS
    qml.AngleEmbedding(
        inputs,
        wires=range(N_QUBITS),
        rotation="Y"
    )

    # APPLY TRAINABLE QUANTUM LAYERS
    qml.StronglyEntanglingLayers(
        weights,
        wires=range(N_QUBITS)
    )

    # MEASURE EACH QUBIT
    return [
        qml.expval(qml.PauliZ(i))
        for i in range(N_QUBITS)
    ]


weight_shapes = {
    "weights": (
        N_Q_LAYERS,
        N_QUBITS,
        3
    )
}

quantum_layer = qml.qnn.TorchLayer(
    quantum_circuit,
    weight_shapes
)

print("Quantum layer created successfully.")
print("Qubits:", N_QUBITS)
print("Quantum layers:", N_Q_LAYERS)

Quantum layer created successfully.
Qubits: 4
Quantum layers: 2


In [7]:
# RECONSTRUCT THE HQNN, LOAD THE SAVED WEIGHTS, AND FREEZE EVERYTHING

N_CLASSES = 4
FEATURE_DIM = 1280

# Create the EfficientNet-B0 structure only.
# weights=None is intentional because all trained weights come from BEST_HQNN.pth.
efficientnet = models.efficientnet_b0(
    weights=None
)


class HQNN(nn.Module):

    def __init__(
        self,
        efficientnet_model,
        quantum_layer
    ):
        super().__init__()

        self.features = (
            efficientnet_model.features
        )

        self.avgpool = (
            efficientnet_model.avgpool
        )

        self.feature_reduction = nn.Linear(
            FEATURE_DIM,
            N_QUBITS
        )

        self.quantum_layer = quantum_layer

        self.classifier = nn.Linear(
            N_QUBITS,
            N_CLASSES
        )

    def forward(self, x):

        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        x = self.feature_reduction(x)
        x = torch.tanh(x) * torch.pi

        x = x.cpu()
        x = self.quantum_layer(x)

        classifier_device = next(
            self.classifier.parameters()
        ).device

        x = x.to(classifier_device)
        x = self.classifier(x)

        return x


hqnn_model = HQNN(
    efficientnet,
    quantum_layer
)

saved_state = torch.load(
    HQNN_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

# strict=True guarantees that the uploaded checkpoint matches
# the exact HQNN architecture used in this experiment.
hqnn_model.load_state_dict(
    saved_state,
    strict=True
)

# HARD-FREEZE THE COMPLETE TRAINED MODEL.
for parameter in hqnn_model.parameters():
    parameter.requires_grad = False

hqnn_model = hqnn_model.cpu()
hqnn_model.eval()

trainable_parameters = sum(
    parameter.numel()
    for parameter in hqnn_model.parameters()
    if parameter.requires_grad
)

assert trainable_parameters == 0, (
    "The loaded HQNN must have zero trainable parameters."
)

# EXTRACT THE LOCKED 24 TRAINED QUANTUM PARAMETERS.
quantum_parameter_tensor = list(
    hqnn_model.quantum_layer.parameters()
)[0]

trained_quantum_weights = (
    quantum_parameter_tensor
    .detach()
    .cpu()
    .reshape(
        N_Q_LAYERS,
        N_QUBITS,
        3
    )
)

flattened_quantum_weights = (
    trained_quantum_weights
    .numpy()
    .reshape(-1)
)

assert len(flattened_quantum_weights) == 24

print("BEST_HQNN checkpoint loaded successfully.")
print("Trainable parameters:", trainable_parameters)
print("Quantum weights:", trained_quantum_weights.shape)
print("Model status: LOCKED FOR INFERENCE ONLY")


BEST_HQNN checkpoint loaded successfully.
Trainable parameters: 0
Quantum weights: torch.Size([2, 4, 3])
Model status: LOCKED FOR INFERENCE ONLY


In [8]:
# CONNECT TO IBM QUANTUM AND LOAD THE CALIBRATION SNAPSHOT

from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

IBM_BACKEND_NAME = "ibm_fez"

ibm_api_key = getpass(
    "Enter IBM Quantum API key: "
)

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=ibm_api_key
)

available_backend_names = {
    backend.name
    for backend in service.backends(
        simulator=False
    )
}

if IBM_BACKEND_NAME not in available_backend_names:
    raise RuntimeError(
        f"{IBM_BACKEND_NAME} is not available to this IBM Quantum account/instance. "
        f"Available real backends include: {sorted(available_backend_names)}"
    )

ibm_backend = service.backend(
    IBM_BACKEND_NAME
)

ibm_backend.refresh()

ibm_properties = ibm_backend.properties(
    refresh=True
)

if ibm_properties is None:
    raise RuntimeError(
        "IBM backend calibration properties are currently unavailable."
    )

print("Connected to IBM Quantum.")
print("Selected backend:", IBM_BACKEND_NAME)
print("Physical qubits:", ibm_backend.num_qubits)
print("Calibration timestamp:", ibm_properties.last_update_date)


Enter IBM Quantum API key: ··········


qiskit_runtime_service._discover_account:WARNING:2026-08-24 04:51:34,044: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-08-24 04:51:38,438: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-08-24 04:51:38,440: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-08-24 04:51:40,023: Using instance: open-instance, plan: open


Connected to IBM Quantum.
Selected backend: ibm_fez
Physical qubits: 156
Calibration timestamp: 2026-08-24 03:06:47+00:00


In [9]:
# FIND LOW-ERROR CONNECTED FOUR-QUBIT REGIONS ON IBM FEZ

import numpy as np


# COLLECT CALIBRATED TWO-QUBIT CONNECTIONS

edge_errors = {}

for instruction_name in ibm_backend.target.keys():

    instruction_map = (
        ibm_backend.target[
            instruction_name
        ]
    )

    if instruction_map is None:
        continue

    for qargs, instruction_properties in (
        instruction_map.items()
    ):

        # SKIP GLOBAL OPERATIONS

        if qargs is None:
            continue

        # KEEP ONLY TWO-QUBIT OPERATIONS

        if len(qargs) != 2:
            continue

        # SKIP OPERATIONS WITHOUT ERROR DATA

        if instruction_properties is None:
            continue

        if instruction_properties.error is None:
            continue

        error = float(
            instruction_properties.error
        )

        # EXCLUDE COMPLETELY UNUSABLE CONNECTIONS

        if error >= 1.0:
            continue

        # STORE THE CONNECTION AS AN UNDIRECTED EDGE

        edge = tuple(
            sorted(qargs)
        )

        # KEEP THE LOWEST AVAILABLE ERROR FOR THE CONNECTION

        if (
            edge not in edge_errors
            or error < edge_errors[edge]
        ):

            edge_errors[edge] = error


# BUILD THE PHYSICAL CONNECTIVITY GRAPH

adjacency = {
    qubit: set()
    for qubit in range(
        ibm_backend.num_qubits
    )
}

for qubit_a, qubit_b in edge_errors:

    adjacency[
        qubit_a
    ].add(
        qubit_b
    )

    adjacency[
        qubit_b
    ].add(
        qubit_a
    )


# FIND CONNECTED GROUPS OF FOUR PHYSICAL QUBITS

connected_groups = set()


def expand_group(
    current_group
):

    if len(
        current_group
    ) == 4:

        connected_groups.add(
            tuple(
                sorted(
                    current_group
                )
            )
        )

        return

    neighbours = set()

    for qubit in current_group:

        neighbours.update(
            adjacency[
                qubit
            ]
        )

    neighbours -= (
        current_group
    )

    for neighbour in neighbours:

        expand_group(
            current_group
            | {neighbour}
        )


for start_qubit in range(
    ibm_backend.num_qubits
):

    expand_group(
        {start_qubit}
    )


# CALCULATE A LOW-ERROR SPANNING TREE FOR EACH REGION

def calculate_mst(
    group
):

    group = set(
        group
    )

    internal_edges = []

    for edge, error in (
        edge_errors.items()
    ):

        if (
            edge[0] in group
            and edge[1] in group
        ):

            internal_edges.append(
                (
                    error,
                    edge[0],
                    edge[1]
                )
            )

    internal_edges.sort()

    parent = {
        qubit: qubit
        for qubit in group
    }


    def find(
        qubit
    ):

        while (
            parent[
                qubit
            ] != qubit
        ):

            parent[
                qubit
            ] = parent[
                parent[
                    qubit
                ]
            ]

            qubit = parent[
                qubit
            ]

        return qubit


    selected_edges = []

    for (
        error,
        qubit_a,
        qubit_b
    ) in internal_edges:

        root_a = find(
            qubit_a
        )

        root_b = find(
            qubit_b
        )

        if root_a != root_b:

            parent[
                root_b
            ] = root_a

            selected_edges.append(
                (
                    qubit_a,
                    qubit_b,
                    error
                )
            )

        if len(
            selected_edges
        ) == 3:

            break

    return selected_edges


# SCORE EACH CONNECTED FOUR-QUBIT REGION

candidate_regions = []

for group in connected_groups:

    mst_edges = calculate_mst(
        group
    )

    if len(
        mst_edges
    ) != 3:
        continue

    two_qubit_errors = [
        edge[2]
        for edge in mst_edges
    ]

    readout_errors = []
    t1_values = []
    t2_values = []

    valid_group = True

    for qubit in group:

        try:

            readout_errors.append(
                ibm_properties.readout_error(
                    qubit
                )
            )

            t1_values.append(
                ibm_properties.t1(
                    qubit
                )
            )

            t2_values.append(
                ibm_properties.t2(
                    qubit
                )
            )

        except Exception:

            valid_group = False
            break

    if not valid_group:
        continue

    candidate_regions.append(
        {
            "qubits": group,

            "mean_2q_error": np.mean(
                two_qubit_errors
            ),

            "max_2q_error": np.max(
                two_qubit_errors
            ),

            "mean_readout_error": np.mean(
                readout_errors
            ),

            "median_t1": np.median(
                t1_values
            ),

            "median_t2": np.median(
                t2_values
            ),

            "mst_edges": mst_edges
        }
    )


# RANK REGIONS BY TWO-QUBIT ERROR FIRST

candidate_regions.sort(
    key=lambda region: (
        region[
            "mean_2q_error"
        ],
        region[
            "mean_readout_error"
        ]
    )
)


# PRINT THE TEN BEST REGIONS

print(
    "TOP 10 CONNECTED FOUR-QUBIT REGIONS ON IBM FEZ"
)

print(
    "------------------------------------------------------------"
)

for rank, region in enumerate(
    candidate_regions[:10],
    start=1
):

    print(
        f"Rank {rank}"
    )

    print(
        "Physical qubits:",
        region[
            "qubits"
        ]
    )

    print(
        f"Mean 2-qubit error: "
        f"{region['mean_2q_error'] * 100:.3f}%"
    )

    print(
        f"Maximum 2-qubit error: "
        f"{region['max_2q_error'] * 100:.3f}%"
    )

    print(
        f"Mean readout error: "
        f"{region['mean_readout_error'] * 100:.3f}%"
    )

    print(
        f"Median T1: "
        f"{region['median_t1'] * 1e6:.2f} us"
    )

    print(
        f"Median T2: "
        f"{region['median_t2'] * 1e6:.2f} us"
    )

    print(
        "Connectivity edges:",
        region[
            "mst_edges"
        ]
    )

    print(
        "------------------------------------------------------------"
    )

TOP 10 CONNECTED FOUR-QUBIT REGIONS ON IBM FEZ
------------------------------------------------------------
Rank 1
Physical qubits: (76, 80, 81, 82)
Mean 2-qubit error: 0.178%
Maximum 2-qubit error: 0.188%
Mean readout error: 1.959%
Median T1: 137.32 us
Median T2: 59.04 us
Connectivity edges: [(81, 82, 0.0015807128886584643), (80, 81, 0.0018751165876941256), (76, 81, 0.0018846718807590812)]
------------------------------------------------------------
Rank 2
Physical qubits: (33, 34, 35, 39)
Mean 2-qubit error: 0.180%
Maximum 2-qubit error: 0.213%
Mean readout error: 1.642%
Median T1: 116.60 us
Median T2: 73.13 us
Connectivity edges: [(33, 39, 0.0014906210716430501), (34, 35, 0.0017706934676902097), (33, 34, 0.002131480679401637)]
------------------------------------------------------------
Rank 3
Physical qubits: (91, 92, 93, 98)
Mean 2-qubit error: 0.180%
Maximum 2-qubit error: 0.187%
Mean readout error: 1.239%
Median T1: 141.39 us
Median T2: 73.05 us
Connectivity edges: [(91, 98, 0.0

In [10]:
# TRANSPILE THE FOUR-QUBIT HQNN CIRCUIT ON THE TOP CANDIDATE REGIONS

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.transpiler import generate_preset_pass_manager


# CREATE SYMBOLIC INPUT PARAMETERS

input_parameters = ParameterVector(
    "x",
    4
)

weight_parameters = ParameterVector(
    "w",
    24
)


# BUILD A QISKIT CIRCUIT WITH THE SAME QUANTUM TOPOLOGY AS THE HQNN

reference_circuit = QuantumCircuit(
    4
)


# APPLY ANGLE EMBEDDING WITH Y ROTATIONS

for qubit in range(
    4
):

    reference_circuit.ry(
        input_parameters[
            qubit
        ],
        qubit
    )


# APPLY THE FIRST STRONGLY ENTANGLING LAYER

parameter_index = 0

for qubit in range(
    4
):

    reference_circuit.rz(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1

    reference_circuit.ry(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1

    reference_circuit.rz(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1


# FIRST LAYER USES RANGE 1

for control in range(
    4
):

    target = (
        control + 1
    ) % 4

    reference_circuit.cx(
        control,
        target
    )


# APPLY THE SECOND STRONGLY ENTANGLING LAYER

for qubit in range(
    4
):

    reference_circuit.rz(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1

    reference_circuit.ry(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1

    reference_circuit.rz(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1


# SECOND LAYER USES RANGE 2

for control in range(
    4
):

    target = (
        control + 2
    ) % 4

    reference_circuit.cx(
        control,
        target
    )


print(
    "Reference HQNN quantum circuit created."
)

print(
    "Logical circuit depth:",
    reference_circuit.depth()
)

print(
    "Logical operation counts:",
    reference_circuit.count_ops()
)


# TRANSPILE THE CIRCUIT ON EACH OF THE TOP TEN REGIONS

transpilation_results = []

for rank, region in enumerate(
    candidate_regions[:10],
    start=1
):

    physical_qubits = list(
        region[
            "qubits"
        ]
    )

    # CREATE A BACKEND-AWARE PASS MANAGER

    pass_manager = generate_preset_pass_manager(
        backend=ibm_backend,
        optimization_level=3,
        initial_layout=physical_qubits,
        seed_transpiler=42
    )

    # TRANSPILE THE REFERENCE CIRCUIT

    transpiled_circuit = pass_manager.run(
        reference_circuit
    )


    # COUNT TWO-QUBIT OPERATIONS AFTER TRANSPILATION

    two_qubit_operation_count = 0

    for instruction in (
        transpiled_circuit.data
    ):

        if len(
            instruction.qubits
        ) == 2:

            two_qubit_operation_count += 1


    # GET THE FINAL PHYSICAL LAYOUT

    final_layout = None

    if (
        transpiled_circuit.layout
        is not None
    ):

        final_layout = (
            transpiled_circuit.layout.final_index_layout(
                filter_ancillas=True
            )
        )


    # STORE RESULTS

    transpilation_results.append(
        {
            "rank": rank,
            "qubits": tuple(
                physical_qubits
            ),
            "depth": transpiled_circuit.depth(),
            "two_qubit_ops": two_qubit_operation_count,
            "total_ops": sum(
                transpiled_circuit.count_ops().values()
            ),
            "mean_2q_error": region[
                "mean_2q_error"
            ],
            "max_2q_error": region[
                "max_2q_error"
            ],
            "readout_error": region[
                "mean_readout_error"
            ],
            "final_layout": final_layout,
            "circuit": transpiled_circuit
        }
    )


# RANK PRIMARILY BY TWO-QUBIT OPERATION COUNT,
# THEN CIRCUIT DEPTH,
# THEN CALIBRATION ERROR

transpilation_results.sort(
    key=lambda result: (
        result[
            "two_qubit_ops"
        ],
        result[
            "depth"
        ],
        result[
            "mean_2q_error"
        ],
        result[
            "readout_error"
        ]
    )
)


# PRINT THE RESULTS

print(
    "\nTRANSPILED FOUR-QUBIT REGION COMPARISON"
)

print(
    "------------------------------------------------------------"
)

for result in (
    transpilation_results
):

    print(
        f"Original calibration rank: "
        f"{result['rank']}"
    )

    print(
        f"Physical qubits: "
        f"{result['qubits']}"
    )

    print(
        f"Transpiled circuit depth: "
        f"{result['depth']}"
    )

    print(
        f"Two-qubit operations: "
        f"{result['two_qubit_ops']}"
    )

    print(
        f"Total operations: "
        f"{result['total_ops']}"
    )

    print(
        f"Mean 2-qubit error: "
        f"{result['mean_2q_error'] * 100:.3f}%"
    )

    print(
        f"Maximum 2-qubit error: "
        f"{result['max_2q_error'] * 100:.3f}%"
    )

    print(
        f"Mean readout error: "
        f"{result['readout_error'] * 100:.3f}%"
    )

    print(
        f"Final physical layout: "
        f"{result['final_layout']}"
    )

    print(
        "------------------------------------------------------------"
    )

Reference HQNN quantum circuit created.
Logical circuit depth: 13
Logical operation counts: OrderedDict({'rz': 16, 'ry': 12, 'cx': 8})

TRANSPILED FOUR-QUBIT REGION COMPARISON
------------------------------------------------------------
Original calibration rank: 2
Physical qubits: (33, 34, 35, 39)
Transpiled circuit depth: 54
Two-qubit operations: 17
Total operations: 135
Mean 2-qubit error: 0.180%
Maximum 2-qubit error: 0.213%
Mean readout error: 1.642%
Final physical layout: [39, 35, 33, 34]
------------------------------------------------------------
Original calibration rank: 3
Physical qubits: (91, 92, 93, 98)
Transpiled circuit depth: 54
Two-qubit operations: 17
Total operations: 135
Mean 2-qubit error: 0.180%
Maximum 2-qubit error: 0.187%
Mean readout error: 1.239%
Final physical layout: [98, 93, 91, 92]
------------------------------------------------------------
Original calibration rank: 4
Physical qubits: (25, 37, 44, 45)
Transpiled circuit depth: 59
Two-qubit operations: 1

In [11]:
# SELECT THE BEST HARDWARE-AWARE FOUR-QUBIT REGION AND BUILD SIMULATORS

from qiskit_aer import AerSimulator

EVALUATION_SHOTS = 1024

if len(transpilation_results) == 0:
    raise RuntimeError(
        "No valid four-qubit transpilation candidates were produced."
    )

# transpilation_results was already sorted by:
# 1) two-qubit operation count
# 2) circuit depth
# 3) mean two-qubit error
# 4) readout error
selected_transpilation = (
    transpilation_results[0]
)

SELECTED_CALIBRATION_RANK = (
    selected_transpilation["rank"]
)

SELECTED_PHYSICAL_REGION = tuple(
    selected_transpilation["qubits"]
)

# Calibration-derived noisy simulator.
noisy_ibm_simulator = (
    AerSimulator.from_backend(
        ibm_backend,
        method="density_matrix",
        enable_truncation=True
    )
)

# Ideal finite-shot control with the same number of shots.
ideal_shot_simulator = AerSimulator(
    method="density_matrix",
    enable_truncation=True
)

print("\nFINAL IBM CALIBRATION CONFIGURATION")
print("-----------------------------------")
print("Backend:", IBM_BACKEND_NAME)
print(
    "Original calibration rank:",
    SELECTED_CALIBRATION_RANK
)
print(
    "Selected physical region:",
    SELECTED_PHYSICAL_REGION
)
print(
    "Transpiled depth:",
    selected_transpilation["depth"]
)
print(
    "Two-qubit operations:",
    selected_transpilation["two_qubit_ops"]
)
print(
    "Mean 2-qubit error:",
    f"{selected_transpilation['mean_2q_error'] * 100:.3f}%"
)
print(
    "Maximum 2-qubit error:",
    f"{selected_transpilation['max_2q_error'] * 100:.3f}%"
)
print(
    "Mean readout error:",
    f"{selected_transpilation['readout_error'] * 100:.3f}%"
)
print(
    "Calibration timestamp:",
    ibm_properties.last_update_date
)
print("Evaluation shots:", EVALUATION_SHOTS)

# Save the exact calibration snapshot used in this run.
CALIBRATION_SNAPSHOT_PATH = (
    "/content/ibm_calibration_snapshot.json"
)

with open(
    CALIBRATION_SNAPSHOT_PATH,
    "w"
) as file:

    json.dump(
        ibm_properties.to_dict(),
        file,
        indent=2,
        default=str
    )

print(
    "Calibration snapshot saved:",
    CALIBRATION_SNAPSHOT_PATH
)



FINAL IBM CALIBRATION CONFIGURATION
-----------------------------------
Backend: ibm_fez
Original calibration rank: 2
Selected physical region: (33, 34, 35, 39)
Transpiled depth: 54
Two-qubit operations: 17
Mean 2-qubit error: 0.180%
Maximum 2-qubit error: 0.213%
Mean readout error: 1.642%
Calibration timestamp: 2026-08-24 03:06:47+00:00
Evaluation shots: 1024
Calibration snapshot saved: /content/ibm_calibration_snapshot.json


In [12]:
# CREATE THE CORRECT MEASURED QISKIT CIRCUIT

from qiskit import ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

# Measure logical qubits BEFORE transpilation.
# This preserves the logical-qubit/classical-bit relationship through routing.
measured_reference_circuit = (
    reference_circuit.copy()
)

measurement_register = ClassicalRegister(
    N_QUBITS,
    "measurement"
)

measured_reference_circuit.add_register(
    measurement_register
)

for logical_qubit in range(N_QUBITS):

    measured_reference_circuit.measure(
        logical_qubit,
        measurement_register[
            logical_qubit
        ]
    )

measurement_pass_manager = (
    generate_preset_pass_manager(
        backend=ibm_backend,
        optimization_level=3,
        initial_layout=list(
            SELECTED_PHYSICAL_REGION
        ),
        seed_transpiler=42
    )
)

measured_transpiled_circuit = (
    measurement_pass_manager.run(
        measured_reference_circuit
    )
)

measured_parameter_lookup = {
    parameter.name: parameter
    for parameter
    in measured_transpiled_circuit.parameters
}

expected_parameter_count = (
    N_QUBITS
    + len(flattened_quantum_weights)
)

assert (
    len(measured_transpiled_circuit.parameters)
    == expected_parameter_count
), (
    "Unexpected number of parameters after transpilation."
)

print("Measured hardware-aware circuit ready.")
print(
    "Circuit depth:",
    measured_transpiled_circuit.depth()
)
print(
    "Classical bits:",
    measured_transpiled_circuit.num_clbits
)
print(
    "Parameters:",
    len(measured_transpiled_circuit.parameters)
)


def counts_to_z_expectations(
    counts,
    number_of_qubits=4
):

    total_shots = sum(
        counts.values()
    )

    expectations = np.zeros(
        number_of_qubits,
        dtype=np.float32
    )

    for bitstring, count in counts.items():

        clean_bitstring = (
            bitstring.replace(" ", "")
        )

        # Qiskit reports classical bit strings in reverse display order.
        bits = clean_bitstring[::-1]

        for qubit in range(
            number_of_qubits
        ):

            if bits[qubit] == "0":
                expectations[qubit] += count
            else:
                expectations[qubit] -= count

    expectations /= total_shots

    return expectations


def execute_quantum_input(
    quantum_input,
    simulator,
    seed
):

    parameter_bindings = {}

    # Four MRI-derived input angles.
    for index in range(
        N_QUBITS
    ):

        parameter_bindings[
            measured_parameter_lookup[
                f"x[{index}]"
            ]
        ] = float(
            quantum_input[index]
        )

    # Twenty-four fixed trained quantum weights.
    for index in range(
        len(flattened_quantum_weights)
    ):

        parameter_bindings[
            measured_parameter_lookup[
                f"w[{index}]"
            ]
        ] = float(
            flattened_quantum_weights[
                index
            ]
        )

    bound_circuit = (
        measured_transpiled_circuit
        .assign_parameters(
            parameter_bindings,
            inplace=False
        )
    )

    job = simulator.run(
        bound_circuit,
        shots=EVALUATION_SHOTS,
        seed_simulator=seed
    )

    counts = (
        job.result().get_counts()
    )

    return counts_to_z_expectations(
        counts,
        N_QUBITS
    )


Measured hardware-aware circuit ready.
Circuit depth: 55
Classical bits: 4
Parameters: 28


In [13]:
# CREATE THE FIXED 40-IMAGE SUBSET AND EXTRACT ITS QUANTUM INPUTS

from torch.utils.data import Subset, DataLoader

SUBSET_PER_CLASS = 10
SUBSET_SEED = 42

rng = np.random.default_rng(
    SUBSET_SEED
)

selected_indices = []

for class_name in classes:

    class_indices = np.array([
        index
        for index, label
        in enumerate(test_labels)
        if label == class_name
    ])

    chosen = rng.choice(
        class_indices,
        size=SUBSET_PER_CLASS,
        replace=False
    )

    selected_indices.extend(
        chosen.tolist()
    )

quantum_test_subset = Subset(
    test_dataset,
    selected_indices
)

quantum_test_loader = DataLoader(
    quantum_test_subset,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

feature_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

hqnn_model.features = (
    hqnn_model.features.to(
        feature_device
    )
)

hqnn_model.avgpool = (
    hqnn_model.avgpool.to(
        feature_device
    )
)

hqnn_model.feature_reduction = (
    hqnn_model.feature_reduction.to(
        feature_device
    )
)

hqnn_model.eval()

quantum_inputs_40 = []
quantum_labels_40 = []

with torch.no_grad():

    for images, labels in (
        quantum_test_loader
    ):

        images = images.to(
            feature_device
        )

        features = (
            hqnn_model.features(
                images
            )
        )

        features = (
            hqnn_model.avgpool(
                features
            )
        )

        features = torch.flatten(
            features,
            1
        )

        quantum_inputs = (
            hqnn_model.feature_reduction(
                features
            )
        )

        quantum_inputs = (
            torch.tanh(
                quantum_inputs
            )
            * torch.pi
        )

        quantum_inputs_40.append(
            quantum_inputs
            .cpu()
            .numpy()[0]
        )

        quantum_labels_40.append(
            labels.item()
        )

quantum_inputs_40 = np.array(
    quantum_inputs_40
)

quantum_labels_40 = np.array(
    quantum_labels_40
)

assert quantum_inputs_40.shape == (
    40,
    N_QUBITS
)

print("Fixed 40-image subset ready.")
print(
    "Quantum input shape:",
    quantum_inputs_40.shape
)
print(
    "Class counts:",
    np.bincount(
        quantum_labels_40
    )
)


Fixed 40-image subset ready.
Quantum input shape: (40, 4)
Class counts: [10 10 10 10]


In [14]:
# VERIFY THAT THE QISKIT CIRCUIT IS THE SAME TRAINED PENNYLANE CIRCUIT

from qiskit.quantum_info import Statevector, Pauli

hqnn_model.quantum_layer = (
    hqnn_model.quantum_layer.cpu()
)

hqnn_model.quantum_layer.eval()

verification_input = (
    quantum_inputs_40[0]
)

verification_tensor = torch.tensor(
    verification_input,
    dtype=torch.float32
).unsqueeze(0)

# ------------------------------------------------------------
# 1. EXACT PENNYLANE EXPECTATION VALUES
# ------------------------------------------------------------

with torch.no_grad():

    pennylane_expectations = (
        hqnn_model.quantum_layer(
            verification_tensor
        )
        .cpu()
        .numpy()[0]
    )

# ------------------------------------------------------------
# 2. EXACT QISKIT STATEVECTOR EXPECTATION VALUES
#    This checks the manually reconstructed quantum topology
#    BEFORE finite-shot sampling or hardware noise is introduced.
# ------------------------------------------------------------

reference_parameter_lookup = {
    parameter.name: parameter
    for parameter
    in reference_circuit.parameters
}

reference_bindings = {}

for index in range(N_QUBITS):

    reference_bindings[
        reference_parameter_lookup[
            f"x[{index}]"
        ]
    ] = float(
        verification_input[index]
    )

for index in range(
    len(flattened_quantum_weights)
):

    reference_bindings[
        reference_parameter_lookup[
            f"w[{index}]"
        ]
    ] = float(
        flattened_quantum_weights[index]
    )

bound_reference_circuit = (
    reference_circuit.assign_parameters(
        reference_bindings,
        inplace=False
    )
)

statevector = Statevector.from_instruction(
    bound_reference_circuit
)

qiskit_exact_expectations = []

for qubit in range(N_QUBITS):

    # Qiskit Pauli strings are ordered q_(n-1) ... q_0.
    pauli_label = ["I"] * N_QUBITS
    pauli_label[
        N_QUBITS - 1 - qubit
    ] = "Z"

    observable = Pauli(
        "".join(pauli_label)
    )

    expectation = (
        statevector.expectation_value(
            observable
        )
    )

    qiskit_exact_expectations.append(
        float(np.real(expectation))
    )

qiskit_exact_expectations = np.array(
    qiskit_exact_expectations,
    dtype=np.float64
)

exact_absolute_difference = np.abs(
    pennylane_expectations
    - qiskit_exact_expectations
)

print("PennyLane exact expectations:")
print(
    np.round(
        pennylane_expectations,
        8
    )
)

print("\nQiskit exact expectations:")
print(
    np.round(
        qiskit_exact_expectations,
        8
    )
)

print("\nExact absolute differences:")
print(
    np.round(
        exact_absolute_difference,
        10
    )
)

assert np.allclose(
    pennylane_expectations,
    qiskit_exact_expectations,
    rtol=1e-6,
    atol=1e-6
), (
    "PennyLane and Qiskit exact circuits are not equivalent. "
    "Stop the experiment and inspect the circuit reconstruction."
)

print(
    "\nEXACT CIRCUIT EQUIVALENCE CHECK: PASSED"
)

# ------------------------------------------------------------
# 3. FINITE-SHOT TRANSPILED SANITY CHECK
# ------------------------------------------------------------

qiskit_shot_expectations = execute_quantum_input(
    verification_input,
    ideal_shot_simulator,
    seed=42
)

shot_absolute_difference = np.abs(
    pennylane_expectations
    - qiskit_shot_expectations
)

print("\nQiskit ideal 1024-shot expectations:")
print(
    np.round(
        qiskit_shot_expectations,
        4
    )
)

print(
    "\nMaximum exact-vs-1024-shot difference:",
    float(
        shot_absolute_difference.max()
    )
)

print(
    "Finite-shot check complete. "
    "Sampling differences are expected here."
)


PennyLane exact expectations:
[ 0.7937602   0.3971973   0.5092121  -0.49908248]

Qiskit exact expectations:
[ 0.79376021  0.39719731  0.50921206 -0.49908246]

Exact absolute differences:
[3.11e-08 7.30e-09 1.79e-08 1.45e-08]

EXACT CIRCUIT EQUIVALENCE CHECK: PASSED

Qiskit ideal 1024-shot expectations:
[ 0.8184  0.4492  0.5391 -0.5234]

Maximum exact-vs-1024-shot difference: 0.05202144384384155
Finite-shot check complete. Sampling differences are expected here.


In [15]:
# ============================================================
# FINAL CONTROLLED CALIBRATION-NOISE EXPERIMENT
# SAME 40 MRI IMAGES
# SAME LOCKED BEST_HQNN WEIGHTS
# SAME 1024 SHOTS
# THREE REPEATS: SEEDS 42, 43, 44
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ------------------------------------------------------------
# KEEP THE TRAINED QUANTUM LAYER AND CLASSIFIER LOCKED ON CPU
# ------------------------------------------------------------

hqnn_model.quantum_layer = hqnn_model.quantum_layer.cpu()
hqnn_model.classifier = hqnn_model.classifier.cpu()

hqnn_model.quantum_layer.eval()
hqnn_model.classifier.eval()

for param in hqnn_model.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# HELPER FUNCTION FOR METRICS
# ------------------------------------------------------------

def calculate_metrics(
    true_labels,
    predictions,
    probabilities
):

    return {
        "accuracy": accuracy_score(
            true_labels,
            predictions
        ),

        "precision": precision_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        ),

        "recall": recall_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        ),

        "f1": f1_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        ),

        "auc": roc_auc_score(
            true_labels,
            probabilities,
            multi_class="ovr",
            average="macro"
        )
    }


# ============================================================
# PART 1 — EXACT NOISE-FREE PENNYLANE CONTROL
# ============================================================

print("Running exact noise-free PennyLane control...\n")

exact_predictions = []
exact_probabilities = []
exact_expectations_all = []


for image_index in range(
    len(quantum_inputs_40)
):

    quantum_input_tensor = torch.tensor(
        quantum_inputs_40[image_index],
        dtype=torch.float32
    ).unsqueeze(0)


    with torch.no_grad():

        # EXACT ANALYTICAL QUANTUM OUTPUT
        exact_expectations = (
            hqnn_model.quantum_layer(
                quantum_input_tensor
            )
        )

        # FINAL LOCKED CLASSIFIER
        exact_logits = (
            hqnn_model.classifier(
                exact_expectations
            )
        )

        exact_probs = torch.softmax(
            exact_logits,
            dim=1
        )


    exact_expectations_all.append(
        exact_expectations
        .cpu()
        .numpy()[0]
    )

    exact_predictions.append(
        int(
            torch.argmax(
                exact_logits,
                dim=1
            ).item()
        )
    )

    exact_probabilities.append(
        exact_probs
        .cpu()
        .numpy()[0]
    )


exact_expectations_all = np.array(
    exact_expectations_all
)

exact_predictions = np.array(
    exact_predictions
)

exact_probabilities = np.array(
    exact_probabilities
)


exact_metrics = calculate_metrics(
    quantum_labels_40,
    exact_predictions,
    exact_probabilities
)


print("EXACT NOISE-FREE PENNYLANE RESULTS")

print(
    f"Accuracy:        "
    f"{exact_metrics['accuracy']:.4f}"
)

print(
    f"Macro Precision: "
    f"{exact_metrics['precision']:.4f}"
)

print(
    f"Macro Recall:    "
    f"{exact_metrics['recall']:.4f}"
)

print(
    f"Macro F1:        "
    f"{exact_metrics['f1']:.4f}"
)

print(
    f"Macro ROC-AUC:   "
    f"{exact_metrics['auc']:.4f}"
)


# ============================================================
# PART 2 — THREE IDEAL + CALIBRATION-NOISE REPEATS
# ============================================================

REPEAT_SEEDS = [
    42,
    43,
    44
]

ideal_repeat_results = []
noisy_repeat_results = []


for repeat_number, seed in enumerate(
    REPEAT_SEEDS,
    start=1
):

    print(
        "\n================================================"
    )

    print(
        f"RUN {repeat_number}/3 — "
        f"SEED {seed}"
    )

    print(
        "================================================"
    )


    ideal_predictions = []
    ideal_probabilities = []
    ideal_expectations_all = []

    noisy_predictions = []
    noisy_probabilities = []
    noisy_expectations_all = []


    # --------------------------------------------------------
    # RUN ALL 40 MRI IMAGES
    # --------------------------------------------------------

    for image_index in range(
        len(quantum_inputs_40)
    ):

        quantum_input = (
            quantum_inputs_40[
                image_index
            ]
        )


        # ----------------------------------------------------
        # IDEAL 1024-SHOT SIMULATION
        # ----------------------------------------------------

        shot_seed = (
            seed * 1000
            + image_index
        )

        ideal_expectations = (
            execute_quantum_input(
                quantum_input,
                ideal_shot_simulator,
                seed=shot_seed
            )
        )


        # ----------------------------------------------------
        # IBM CALIBRATION-NOISE 1024-SHOT SIMULATION
        # ----------------------------------------------------

        noisy_expectations = (
            execute_quantum_input(
                quantum_input,
                noisy_ibm_simulator,
                seed=shot_seed
            )
        )


        ideal_expectations_all.append(
            ideal_expectations
        )

        noisy_expectations_all.append(
            noisy_expectations
        )


        # ----------------------------------------------------
        # PASS QUANTUM OUTPUT THROUGH SAME LOCKED CLASSIFIER
        # ----------------------------------------------------

        ideal_tensor = torch.tensor(
            ideal_expectations,
            dtype=torch.float32
        ).unsqueeze(0)

        noisy_tensor = torch.tensor(
            noisy_expectations,
            dtype=torch.float32
        ).unsqueeze(0)


        with torch.no_grad():

            ideal_logits = (
                hqnn_model.classifier(
                    ideal_tensor
                )
            )

            noisy_logits = (
                hqnn_model.classifier(
                    noisy_tensor
                )
            )


            ideal_probs = torch.softmax(
                ideal_logits,
                dim=1
            )

            noisy_probs = torch.softmax(
                noisy_logits,
                dim=1
            )


        ideal_predictions.append(
            int(
                torch.argmax(
                    ideal_logits,
                    dim=1
                ).item()
            )
        )

        noisy_predictions.append(
            int(
                torch.argmax(
                    noisy_logits,
                    dim=1
                ).item()
            )
        )


        ideal_probabilities.append(
            ideal_probs
            .cpu()
            .numpy()[0]
        )

        noisy_probabilities.append(
            noisy_probs
            .cpu()
            .numpy()[0]
        )


    # --------------------------------------------------------
    # CONVERT TO NUMPY
    # --------------------------------------------------------

    ideal_predictions = np.array(
        ideal_predictions
    )

    noisy_predictions = np.array(
        noisy_predictions
    )

    ideal_probabilities = np.array(
        ideal_probabilities
    )

    noisy_probabilities = np.array(
        noisy_probabilities
    )

    ideal_expectations_all = np.array(
        ideal_expectations_all
    )

    noisy_expectations_all = np.array(
        noisy_expectations_all
    )


    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    ideal_metrics = calculate_metrics(
        quantum_labels_40,
        ideal_predictions,
        ideal_probabilities
    )

    noisy_metrics = calculate_metrics(
        quantum_labels_40,
        noisy_predictions,
        noisy_probabilities
    )


    # --------------------------------------------------------
    # DIRECT QUANTUM NOISE EFFECT
    # --------------------------------------------------------

    mean_quantum_difference = np.mean(
        np.abs(
            ideal_expectations_all
            - noisy_expectations_all
        )
    )


    changed_predictions = int(
        np.sum(
            ideal_predictions
            != noisy_predictions
        )
    )


    # --------------------------------------------------------
    # SAVE THIS REPEAT
    # --------------------------------------------------------

    ideal_repeat_results.append(
        {
            **ideal_metrics,

            "quantum_difference":
                np.mean(
                    np.abs(
                        exact_expectations_all
                        - ideal_expectations_all
                    )
                )
        }
    )


    noisy_repeat_results.append(
        {
            **noisy_metrics,

            "quantum_difference":
                mean_quantum_difference,

            "changed_predictions":
                changed_predictions
        }
    )


    # --------------------------------------------------------
    # PRINT THIS REPEAT
    # --------------------------------------------------------

    print(
        "\nIDEAL FINITE-SHOT"
    )

    print(
        f"Accuracy: "
        f"{ideal_metrics['accuracy']:.4f}"
    )

    print(
        f"Macro F1: "
        f"{ideal_metrics['f1']:.4f}"
    )

    print(
        f"ROC-AUC: "
        f"{ideal_metrics['auc']:.4f}"
    )


    print(
        "\nIBM CALIBRATION-NOISE"
    )

    print(
        f"Accuracy: "
        f"{noisy_metrics['accuracy']:.4f}"
    )

    print(
        f"Macro F1: "
        f"{noisy_metrics['f1']:.4f}"
    )

    print(
        f"ROC-AUC: "
        f"{noisy_metrics['auc']:.4f}"
    )


    print(
        "\nMean |Ideal - Noisy| quantum difference:",
        round(
            float(
                mean_quantum_difference
            ),
            4
        )
    )

    print(
        "Predictions changed:",
        f"{changed_predictions}/40"
    )


# ============================================================
# PART 3 — FINAL MEAN ± STANDARD DEVIATION
# ============================================================

ideal_accuracies = np.array([
    result["accuracy"]
    for result in ideal_repeat_results
])

noisy_accuracies = np.array([
    result["accuracy"]
    for result in noisy_repeat_results
])


ideal_f1_scores = np.array([
    result["f1"]
    for result in ideal_repeat_results
])

noisy_f1_scores = np.array([
    result["f1"]
    for result in noisy_repeat_results
])


ideal_auc_scores = np.array([
    result["auc"]
    for result in ideal_repeat_results
])

noisy_auc_scores = np.array([
    result["auc"]
    for result in noisy_repeat_results
])


noise_differences = np.array([
    result["quantum_difference"]
    for result in noisy_repeat_results
])


prediction_changes = np.array([
    result["changed_predictions"]
    for result in noisy_repeat_results
])


print(
    "\n\n================================================"
)

print(
    "FINAL THREE-RUN SUMMARY"
)

print(
    "================================================"
)


print(
    "\nEXACT NOISE-FREE PENNYLANE"
)

print(
    f"Accuracy: "
    f"{exact_metrics['accuracy']:.4f}"
)

print(
    f"Macro F1: "
    f"{exact_metrics['f1']:.4f}"
)

print(
    f"ROC-AUC: "
    f"{exact_metrics['auc']:.4f}"
)


print(
    "\nIDEAL 1024-SHOT SIMULATION"
)

print(
    f"Accuracy: "
    f"{ideal_accuracies.mean():.4f} "
    f"± {ideal_accuracies.std(ddof=1):.4f}"
)

print(
    f"Macro F1: "
    f"{ideal_f1_scores.mean():.4f} "
    f"± {ideal_f1_scores.std(ddof=1):.4f}"
)

print(
    f"ROC-AUC: "
    f"{ideal_auc_scores.mean():.4f} "
    f"± {ideal_auc_scores.std(ddof=1):.4f}"
)


print(
    "\nIBM CALIBRATION-NOISE 1024-SHOT SIMULATION"
)

print(
    f"Accuracy: "
    f"{noisy_accuracies.mean():.4f} "
    f"± {noisy_accuracies.std(ddof=1):.4f}"
)

print(
    f"Macro F1: "
    f"{noisy_f1_scores.mean():.4f} "
    f"± {noisy_f1_scores.std(ddof=1):.4f}"
)

print(
    f"ROC-AUC: "
    f"{noisy_auc_scores.mean():.4f} "
    f"± {noisy_auc_scores.std(ddof=1):.4f}"
)


print(
    "\nNOISE EFFECT"
)

print(
    "Mean absolute ideal-vs-noisy "
    "quantum-output difference:"
)

print(
    f"{noise_differences.mean():.4f} "
    f"± {noise_differences.std(ddof=1):.4f}"
)

print(
    "Mean number of changed predictions:"
)

print(
    f"{prediction_changes.mean():.2f}/40"
)

Running exact noise-free PennyLane control...

EXACT NOISE-FREE PENNYLANE RESULTS
Accuracy:        0.9250
Macro Precision: 0.9295
Macro Recall:    0.9250
Macro F1:        0.9234
Macro ROC-AUC:   0.9875

RUN 1/3 — SEED 42

IDEAL FINITE-SHOT
Accuracy: 0.9250
Macro F1: 0.9234
ROC-AUC: 0.9883

IBM CALIBRATION-NOISE
Accuracy: 0.9250
Macro F1: 0.9234
ROC-AUC: 0.9892

Mean |Ideal - Noisy| quantum difference: 0.064
Predictions changed: 0/40

RUN 2/3 — SEED 43

IDEAL FINITE-SHOT
Accuracy: 0.9250
Macro F1: 0.9234
ROC-AUC: 0.9875

IBM CALIBRATION-NOISE
Accuracy: 0.9250
Macro F1: 0.9234
ROC-AUC: 0.9883

Mean |Ideal - Noisy| quantum difference: 0.0639
Predictions changed: 0/40

RUN 3/3 — SEED 44

IDEAL FINITE-SHOT
Accuracy: 0.9250
Macro F1: 0.9234
ROC-AUC: 0.9867

IBM CALIBRATION-NOISE
Accuracy: 0.9250
Macro F1: 0.9234
ROC-AUC: 0.9850

Mean |Ideal - Noisy| quantum difference: 0.0648
Predictions changed: 0/40


FINAL THREE-RUN SUMMARY

EXACT NOISE-FREE PENNYLANE
Accuracy: 0.9250
Macro F1: 0.9234
ROC

In [16]:
# ============================================================
# EXTRACT QUANTUM INPUTS FOR THE FULL 1,080-IMAGE TEST SET
# ============================================================

import numpy as np
import torch

feature_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

hqnn_model.features = hqnn_model.features.to(feature_device)
hqnn_model.avgpool = hqnn_model.avgpool.to(feature_device)
hqnn_model.feature_reduction = hqnn_model.feature_reduction.to(feature_device)

hqnn_model.eval()

full_quantum_inputs = []
full_test_labels = []

print("Extracting quantum inputs from all test MRIs...")

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(feature_device)

        # EFFICIENTNET-B0
        features = hqnn_model.features(images)

        # GLOBAL AVERAGE POOL
        features = hqnn_model.avgpool(features)

        # 1280 FEATURES
        features = torch.flatten(
            features,
            1
        )

        # 1280 -> 4
        quantum_inputs = (
            hqnn_model.feature_reduction(
                features
            )
        )

        # SAME QUANTUM ANGLE SCALING AS TRAINED HQNN
        quantum_inputs = (
            torch.tanh(
                quantum_inputs
            )
            * torch.pi
        )

        full_quantum_inputs.append(
            quantum_inputs
            .cpu()
            .numpy()
        )

        full_test_labels.append(
            labels
            .cpu()
            .numpy()
        )


full_quantum_inputs = np.concatenate(
    full_quantum_inputs,
    axis=0
)

full_test_labels = np.concatenate(
    full_test_labels,
    axis=0
)


print(
    "Quantum input shape:",
    full_quantum_inputs.shape
)

print(
    "Label shape:",
    full_test_labels.shape
)

print(
    "Class counts:",
    np.bincount(
        full_test_labels
    )
)

Extracting quantum inputs from all test MRIs...
Quantum input shape: (1080, 4)
Label shape: (1080,)
Class counts: [270 270 270 270]


In [17]:
# ============================================================
# FULL 1,080-IMAGE IDEAL VS IBM CALIBRATION-NOISE EXPERIMENT
#
# SAME:
# - BEST_HQNN weights
# - 1,080 test images
# - quantum circuit
# - final classifier
# - 1024 shots
#
# ONLY DIFFERENCE:
# ideal simulator vs IBM calibration-derived noisy simulator
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


# ------------------------------------------------------------
# LOCK FINAL CLASSIFIER
# ------------------------------------------------------------

hqnn_model.classifier = (
    hqnn_model.classifier.cpu()
)

hqnn_model.classifier.eval()

for param in hqnn_model.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# BUILD ALL 1,080 PARAMETER-BOUND QUANTUM CIRCUITS
# ------------------------------------------------------------

print(
    "Building 1,080 trained HQNN quantum circuits..."
)

full_bound_circuits = []


for image_index in range(
    len(full_quantum_inputs)
):

    quantum_input = (
        full_quantum_inputs[
            image_index
        ]
    )

    parameter_bindings = {}


    # 4 MRI-DERIVED QUANTUM INPUTS
    for index in range(
        N_QUBITS
    ):

        parameter_bindings[
            measured_parameter_lookup[
                f"x[{index}]"
            ]
        ] = float(
            quantum_input[index]
        )


    # 24 LOCKED TRAINED QUANTUM WEIGHTS
    for index in range(
        len(
            flattened_quantum_weights
        )
    ):

        parameter_bindings[
            measured_parameter_lookup[
                f"w[{index}]"
            ]
        ] = float(
            flattened_quantum_weights[
                index
            ]
        )


    bound_circuit = (
        measured_transpiled_circuit
        .assign_parameters(
            parameter_bindings,
            inplace=False
        )
    )


    full_bound_circuits.append(
        bound_circuit
    )


print(
    "Circuits prepared:",
    len(full_bound_circuits)
)


# ------------------------------------------------------------
# RUN THE FULL TEST SET IN CHUNKS
# ------------------------------------------------------------

CIRCUIT_CHUNK_SIZE = 100

ideal_expectations_full = []
noisy_expectations_full = []

number_of_chunks = (
    len(full_bound_circuits)
    + CIRCUIT_CHUNK_SIZE
    - 1
) // CIRCUIT_CHUNK_SIZE

print(
    f"Running {len(full_bound_circuits)} circuits "
    f"in {number_of_chunks} chunks..."
)

for chunk_index, start_index in enumerate(
    range(
        0,
        len(full_bound_circuits),
        CIRCUIT_CHUNK_SIZE
    )
):

    end_index = min(
        start_index + CIRCUIT_CHUNK_SIZE,
        len(full_bound_circuits)
    )

    circuit_chunk = (
        full_bound_circuits[
            start_index:end_index
        ]
    )

    chunk_seed = (
        4200 + chunk_index
    )

    print(
        f"Chunk {chunk_index + 1}/{number_of_chunks}: "
        f"images {start_index}–{end_index - 1}"
    )

    # SAME SHOT COUNT AND REPRODUCIBLE SEED FOR BOTH CONDITIONS.
    ideal_job = ideal_shot_simulator.run(
        circuit_chunk,
        shots=EVALUATION_SHOTS,
        seed_simulator=chunk_seed
    )

    noisy_job = noisy_ibm_simulator.run(
        circuit_chunk,
        shots=EVALUATION_SHOTS,
        seed_simulator=chunk_seed
    )

    ideal_result = ideal_job.result()
    noisy_result = noisy_job.result()

    for local_index in range(
        len(circuit_chunk)
    ):

        ideal_counts = (
            ideal_result.get_counts(
                local_index
            )
        )

        noisy_counts = (
            noisy_result.get_counts(
                local_index
            )
        )

        ideal_expectations_full.append(
            counts_to_z_expectations(
                ideal_counts,
                N_QUBITS
            )
        )

        noisy_expectations_full.append(
            counts_to_z_expectations(
                noisy_counts,
                N_QUBITS
            )
        )

ideal_expectations_full = np.array(
    ideal_expectations_full
)

noisy_expectations_full = np.array(
    noisy_expectations_full
)

assert ideal_expectations_full.shape == (
    len(full_test_labels),
    N_QUBITS
)

assert noisy_expectations_full.shape == (
    len(full_test_labels),
    N_QUBITS
)

print(
    "\nIdeal expectation shape:",
    ideal_expectations_full.shape
)

print(
    "Noisy expectation shape:",
    noisy_expectations_full.shape
)


# ------------------------------------------------------------
# PASS BOTH THROUGH THE SAME LOCKED 4 -> 4 CLASSIFIER
# ------------------------------------------------------------

ideal_tensor_full = torch.tensor(
    ideal_expectations_full,
    dtype=torch.float32
)

noisy_tensor_full = torch.tensor(
    noisy_expectations_full,
    dtype=torch.float32
)


with torch.no_grad():

    ideal_logits_full = (
        hqnn_model.classifier(
            ideal_tensor_full
        )
    )

    noisy_logits_full = (
        hqnn_model.classifier(
            noisy_tensor_full
        )
    )


    ideal_probabilities_full = (
        torch.softmax(
            ideal_logits_full,
            dim=1
        )
        .cpu()
        .numpy()
    )

    noisy_probabilities_full = (
        torch.softmax(
            noisy_logits_full,
            dim=1
        )
        .cpu()
        .numpy()
    )


ideal_predictions_full = (
    torch.argmax(
        ideal_logits_full,
        dim=1
    )
    .cpu()
    .numpy()
)

noisy_predictions_full = (
    torch.argmax(
        noisy_logits_full,
        dim=1
    )
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# METRIC FUNCTION
# ------------------------------------------------------------

def calculate_full_metrics(
    labels,
    predictions,
    probabilities
):

    return {

        "accuracy":
            accuracy_score(
                labels,
                predictions
            ),

        "precision":
            precision_score(
                labels,
                predictions,
                average="macro",
                zero_division=0
            ),

        "recall":
            recall_score(
                labels,
                predictions,
                average="macro",
                zero_division=0
            ),

        "f1":
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0
            ),

        "auc":
            roc_auc_score(
                labels,
                probabilities,
                multi_class="ovr",
                average="macro"
            )
    }


ideal_full_metrics = (
    calculate_full_metrics(
        full_test_labels,
        ideal_predictions_full,
        ideal_probabilities_full
    )
)

noisy_full_metrics = (
    calculate_full_metrics(
        full_test_labels,
        noisy_predictions_full,
        noisy_probabilities_full
    )
)


# ------------------------------------------------------------
# DIRECT NOISE EFFECT
# ------------------------------------------------------------

mean_noise_shift_full = np.mean(
    np.abs(
        ideal_expectations_full
        - noisy_expectations_full
    )
)

changed_predictions_full = np.sum(
    ideal_predictions_full
    != noisy_predictions_full
)


# DID NOISE HELP OR HURT EACH CHANGED SAMPLE?
ideal_correct = (
    ideal_predictions_full
    == full_test_labels
)

noisy_correct = (
    noisy_predictions_full
    == full_test_labels
)


noise_fixed_errors = np.sum(
    (~ideal_correct)
    &
    noisy_correct
)

noise_created_errors = np.sum(
    ideal_correct
    &
    (~noisy_correct)
)


# ------------------------------------------------------------
# PRINT FINAL RESULTS
# ------------------------------------------------------------

print(
    "\n========================================"
)

print(
    "FULL 1,080-IMAGE RESULTS"
)

print(
    "========================================"
)


print(
    "\nIDEAL 1024-SHOT SIMULATION"
)

print(
    f"Accuracy:        "
    f"{ideal_full_metrics['accuracy']:.4f}"
)

print(
    f"Macro Precision: "
    f"{ideal_full_metrics['precision']:.4f}"
)

print(
    f"Macro Recall:    "
    f"{ideal_full_metrics['recall']:.4f}"
)

print(
    f"Macro F1:        "
    f"{ideal_full_metrics['f1']:.4f}"
)

print(
    f"Macro ROC-AUC:   "
    f"{ideal_full_metrics['auc']:.4f}"
)


print(
    "\nIBM CALIBRATION-NOISE "
    "1024-SHOT SIMULATION"
)

print(
    f"Accuracy:        "
    f"{noisy_full_metrics['accuracy']:.4f}"
)

print(
    f"Macro Precision: "
    f"{noisy_full_metrics['precision']:.4f}"
)

print(
    f"Macro Recall:    "
    f"{noisy_full_metrics['recall']:.4f}"
)

print(
    f"Macro F1:        "
    f"{noisy_full_metrics['f1']:.4f}"
)

print(
    f"Macro ROC-AUC:   "
    f"{noisy_full_metrics['auc']:.4f}"
)


print(
    "\nNOISE EFFECT"
)

print(
    "Mean absolute quantum-output shift:",
    round(
        float(
            mean_noise_shift_full
        ),
        4
    )
)

print(
    "Predictions changed:",
    f"{changed_predictions_full}/1080"
)

print(
    "Errors FIXED by noise:",
    int(
        noise_fixed_errors
    )
)

print(
    "Errors CREATED by noise:",
    int(
        noise_created_errors
    )
)


print(
    "\nIDEAL CONFUSION MATRIX:"
)

print(
    confusion_matrix(
        full_test_labels,
        ideal_predictions_full
    )
)


print(
    "\nCALIBRATION-NOISE CONFUSION MATRIX:"
)

print(
    confusion_matrix(
        full_test_labels,
        noisy_predictions_full
    )
)

Building 1,080 trained HQNN quantum circuits...
Circuits prepared: 1080
Running 1080 circuits in 11 chunks...
Chunk 1/11: images 0–99
Chunk 2/11: images 100–199
Chunk 3/11: images 200–299
Chunk 4/11: images 300–399
Chunk 5/11: images 400–499
Chunk 6/11: images 500–599
Chunk 7/11: images 600–699
Chunk 8/11: images 700–799
Chunk 9/11: images 800–899
Chunk 10/11: images 900–999
Chunk 11/11: images 1000–1079

Ideal expectation shape: (1080, 4)
Noisy expectation shape: (1080, 4)

FULL 1,080-IMAGE RESULTS

IDEAL 1024-SHOT SIMULATION
Accuracy:        0.9333
Macro Precision: 0.9332
Macro Recall:    0.9333
Macro F1:        0.9328
Macro ROC-AUC:   0.9886

IBM CALIBRATION-NOISE 1024-SHOT SIMULATION
Accuracy:        0.9333
Macro Precision: 0.9332
Macro Recall:    0.9333
Macro F1:        0.9328
Macro ROC-AUC:   0.9886

NOISE EFFECT
Mean absolute quantum-output shift: 0.0644
Predictions changed: 2/1080
Errors FIXED by noise: 1
Errors CREATED by noise: 1

IDEAL CONFUSION MATRIX:
[[249  14   7   0]
 [

In [18]:
# ============================================================
# EXACT NOISE-FREE PENNYLANE CONTROL — FULL 1,080 TEST IMAGES
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


hqnn_model.quantum_layer = hqnn_model.quantum_layer.cpu()
hqnn_model.classifier = hqnn_model.classifier.cpu()

hqnn_model.quantum_layer.eval()
hqnn_model.classifier.eval()


exact_expectations_full = []
exact_predictions_full = []
exact_probabilities_full = []


print("Running exact PennyLane control on 1,080 images...")


with torch.no_grad():

    for image_index in range(
        len(full_quantum_inputs)
    ):

        quantum_input = torch.tensor(
            full_quantum_inputs[image_index],
            dtype=torch.float32
        ).unsqueeze(0)


        # EXACT NOISE-FREE QUANTUM CIRCUIT
        exact_expectations = (
            hqnn_model.quantum_layer(
                quantum_input
            )
        )


        # SAME LOCKED FINAL CLASSIFIER
        exact_logits = (
            hqnn_model.classifier(
                exact_expectations
            )
        )


        exact_probs = torch.softmax(
            exact_logits,
            dim=1
        )


        exact_expectations_full.append(
            exact_expectations
            .cpu()
            .numpy()[0]
        )


        exact_predictions_full.append(
            int(
                torch.argmax(
                    exact_logits,
                    dim=1
                ).item()
            )
        )


        exact_probabilities_full.append(
            exact_probs
            .cpu()
            .numpy()[0]
        )


exact_expectations_full = np.array(
    exact_expectations_full
)

exact_predictions_full = np.array(
    exact_predictions_full
)

exact_probabilities_full = np.array(
    exact_probabilities_full
)


# ============================================================
# METRICS
# ============================================================

exact_accuracy_full = accuracy_score(
    full_test_labels,
    exact_predictions_full
)

exact_precision_full = precision_score(
    full_test_labels,
    exact_predictions_full,
    average="macro",
    zero_division=0
)

exact_recall_full = recall_score(
    full_test_labels,
    exact_predictions_full,
    average="macro",
    zero_division=0
)

exact_f1_full = f1_score(
    full_test_labels,
    exact_predictions_full,
    average="macro",
    zero_division=0
)

exact_auc_full = roc_auc_score(
    full_test_labels,
    exact_probabilities_full,
    multi_class="ovr",
    average="macro"
)


print(
    "\n========================================"
)

print(
    "EXACT NOISE-FREE PENNYLANE — 1,080 IMAGES"
)

print(
    "========================================"
)

print(
    f"Accuracy:        {exact_accuracy_full:.4f}"
)

print(
    f"Macro Precision: {exact_precision_full:.4f}"
)

print(
    f"Macro Recall:    {exact_recall_full:.4f}"
)

print(
    f"Macro F1:        {exact_f1_full:.4f}"
)

print(
    f"Macro ROC-AUC:   {exact_auc_full:.4f}"
)


# ============================================================
# DECOMPOSE THE PERFORMANCE LOSS
# ============================================================

print(
    "\n========================================"
)

print(
    "PERFORMANCE DECOMPOSITION"
)

print(
    "========================================"
)


print(
    "\nExact noise-free accuracy:",
    f"{exact_accuracy_full:.4f}"
)

print(
    "Ideal 1024-shot accuracy:",
    f"{ideal_full_metrics['accuracy']:.4f}"
)

print(
    "Calibration-noise accuracy:",
    f"{noisy_full_metrics['accuracy']:.4f}"
)


print(
    "\nLoss from finite-shot sampling:",
    f"{exact_accuracy_full - ideal_full_metrics['accuracy']:.4f}"
)

print(
    "Additional loss from calibration noise:",
    f"{ideal_full_metrics['accuracy'] - noisy_full_metrics['accuracy']:.4f}"
)

print(
    "Total loss from exact to noisy:",
    f"{exact_accuracy_full - noisy_full_metrics['accuracy']:.4f}"
)

Running exact PennyLane control on 1,080 images...

EXACT NOISE-FREE PENNYLANE — 1,080 IMAGES
Accuracy:        0.9315
Macro Precision: 0.9313
Macro Recall:    0.9315
Macro F1:        0.9309
Macro ROC-AUC:   0.9888

PERFORMANCE DECOMPOSITION

Exact noise-free accuracy: 0.9315
Ideal 1024-shot accuracy: 0.9333
Calibration-noise accuracy: 0.9333

Loss from finite-shot sampling: -0.0019
Additional loss from calibration noise: 0.0000
Total loss from exact to noisy: -0.0019


In [19]:
# SAVE AND DOWNLOAD THE FINAL CALIBRATION-NOISE RESULTS

import pandas as pd
import hashlib
from google.colab import files

results_table = pd.DataFrame([
    {
        "condition": "Exact noise-free PennyLane",
        "shots": "exact",
        "accuracy": exact_accuracy_full,
        "macro_precision": exact_precision_full,
        "macro_recall": exact_recall_full,
        "macro_f1": exact_f1_full,
        "macro_roc_auc": exact_auc_full
    },
    {
        "condition": "Ideal finite-shot Qiskit",
        "shots": EVALUATION_SHOTS,
        "accuracy": ideal_full_metrics["accuracy"],
        "macro_precision": ideal_full_metrics["precision"],
        "macro_recall": ideal_full_metrics["recall"],
        "macro_f1": ideal_full_metrics["f1"],
        "macro_roc_auc": ideal_full_metrics["auc"]
    },
    {
        "condition": "IBM calibration-derived noise",
        "shots": EVALUATION_SHOTS,
        "accuracy": noisy_full_metrics["accuracy"],
        "macro_precision": noisy_full_metrics["precision"],
        "macro_recall": noisy_full_metrics["recall"],
        "macro_f1": noisy_full_metrics["f1"],
        "macro_roc_auc": noisy_full_metrics["auc"]
    }
])

RESULTS_PATH = (
    "/content/HQNN_calibration_noise_results.csv"
)

results_table.to_csv(
    RESULTS_PATH,
    index=False
)

print(results_table)

print(
    "\nMean absolute ideal-vs-noisy quantum-output shift:",
    float(mean_noise_shift_full)
)

print(
    "Changed final predictions:",
    int(changed_predictions_full)
)


# Save the exact test split used in this calibration run.
test_split_table = pd.DataFrame({
    "file": test_files,
    "label": test_labels
})

TEST_SPLIT_PATH = (
    "/content/HQNN_calibration_test_split.csv"
)

test_split_table.to_csv(
    TEST_SPLIT_PATH,
    index=False
)

# Create a compact fingerprint for the exact ordered test split.
fingerprint_payload = "\n".join(
    f"{file_path}|{label}"
    for file_path, label
    in zip(test_files, test_labels)
)

test_split_fingerprint = hashlib.sha256(
    fingerprint_payload.encode("utf-8")
).hexdigest()

print(
    "\nTest split SHA-256 fingerprint:",
    test_split_fingerprint
)

print("\nDownloading result files...")

files.download(
    RESULTS_PATH
)

files.download(
    CALIBRATION_SNAPSHOT_PATH
)

files.download(
    TEST_SPLIT_PATH
)


                       condition  shots  accuracy  macro_precision  \
0     Exact noise-free PennyLane  exact  0.931481         0.931306   
1       Ideal finite-shot Qiskit   1024  0.933333         0.933218   
2  IBM calibration-derived noise   1024  0.933333         0.933218   

   macro_recall  macro_f1  macro_roc_auc  
0      0.931481  0.930927       0.988837  
1      0.933333  0.932808       0.988628  
2      0.933333  0.932808       0.988639  

Mean absolute ideal-vs-noisy quantum-output shift: 0.0644192174077034
Changed final predictions: 2

Test split SHA-256 fingerprint: 82958012c99477ae253314c8f71f017a03342285e54e5e0603ce25b3cb37beaf



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>